# Lesson 16 — Dataclass (SUPPORTING)

Run the cells top to bottom. For every exercise you create the files
yourself — write your work in `solution.py` first, then create the files
each exercise lists, then run the **Driver:** cell. The theme is the
same Expense/Transaction domain, rebuilt as data classes — this model
transfers into Weekly Project 2.

If a driver shows no output, check that the file names and locations
match the exercise's **File tree:** byte-for-byte.

Read the **Output:** block of each exercise *before* you run its driver.
Predict, then run.

## E1 — `@dataclass` baseline: annotations become the code

**Type:** [REAL-WORLD] · easy

**Task:** create `record.py` with one dataclass:

- `@dataclass class Transaction` — annotated **fields only**: `date`
  (`str`), `category` (`str`), `kind` (`str`), `amount` (`float`).
- Method `signed_amount(self) -> float`: returns `+amount` when `kind`
  is `"income"`, otherwise `-amount`.

You do **not** write `__init__`, `__repr__`, or `__eq__` — the
decorator generates them from the fields. Compare with the hand-written
`transaction.py` from Lesson 15 E1: same class, minus the boilerplate.

**Constraints:**
- Every field is annotated; every method is annotated (`signed_amount`
  returns `float`).
- `signed_amount` reads fields through `self` — it is a normal class
  method.

**File tree:**
```text
record.py           <- you create
```

**Driver:**
```python
from typing import get_type_hints
from record import Transaction

food = Transaction("2026-09-04", "food", "expense", 12.5)
salary = Transaction("2026-09-25", "salary", "income", 100.0)

print(food)
print(food.signed_amount())
print(salary.signed_amount())
print(food == Transaction("2026-09-04", "food", "expense", 12.5))
print(food is salary)
print(sorted(get_type_hints(Transaction)))
print({k: v.__name__ for k, v in get_type_hints(Transaction.signed_amount).items()})
```

**Output:**
```
Transaction(date='2026-09-04', category='food', kind='expense', amount=12.5)
-12.5
100.0
True
False
['amount', 'category', 'date', 'kind']
{'return': 'float'}
```

In [1]:
from typing import get_type_hints
from record import Transaction

food = Transaction("2026-09-04", "food", "expense", 12.5)
salary = Transaction("2026-09-25", "salary", "income", 100.0)

print(food)
print(food.signed_amount())
print(salary.signed_amount())
print(food == Transaction("2026-09-04", "food", "expense", 12.5))
print(food is salary)
print(sorted(get_type_hints(Transaction)))
print({k: v.__name__ for k, v in get_type_hints(Transaction.signed_amount).items()})

Transaction(date='2026-09-04', category='food', kind='expense', amount=12.5)
-12.5
100.0
True
False
['amount', 'category', 'date', 'kind']
{'return': 'float'}


## E2 — defaults: plain value vs `field(default_factory=...)`

**Type:** [REAL-WORLD] · easy

**Task:** create `category.py` with one dataclass:

- `@dataclass class Category` — fields: `name` (`str`),
  `budget` (`float = 0.0`), and `tags` (`list[str]` — a **mutable**
  field with `field(default_factory=list)`).

Why the factory? Try `tags: list[str] = []` and watch the import fail
with `ValueError: mutable default <class 'list'> ... not allowed`. Same
shared-object trap as Lesson 15, but the dataclass makes it loud at
class-definition time instead of silent at runtime. `default_factory`
calls `list()` fresh for every instance.

**Constraints:**
- `from dataclasses import dataclass, field`.
- Fields with defaults come after fields without them (`budget` and
  `tags` after `name`).

**File tree:**
```text
category.py        <- you create
```

**Driver:**
```python
from category import Category

food = Category("food", 300.0)
ride = Category("ride")

print(food)
print(ride)
print(ride.budget)
food.tags.append("everyday")
print(food.tags)
print(ride.tags)
print(food.tags is not ride.tags)
print(Category("ride") == ride)
```

**Output:**
```
Category(name='food', budget=300.0, tags=[])
Category(name='ride', budget=0.0, tags=[])
0.0
['everyday']
[]
True
True
```

In [2]:
from category import Category

food = Category("food", 300.0)
ride = Category("ride")

print(food)
print(ride)
print(ride.budget)
food.tags.append("everyday")
print(food.tags)
print(ride.tags)
print(food.tags is not ride.tags)
print(Category("ride") == ride)

Category(name='food', budget=300.0, tags=[])
Category(name='ride', budget=0.0, tags=[])
0.0
['everyday']
[]
True
True


## E3 — `frozen=True` + the `Kind` Enum (fold from Lesson 15 E4)

**Type:** [REAL-WORLD] · easy

**Task:** create `kind.py` with:

- `class Kind(Enum)` — members `INCOME = "income"` and
  `EXPENSE = "expense"` (reuse from Lesson 15 E4).
- `@dataclass(frozen=True) class Transaction` — fields `date` (`str`),
  `category` (`str`), `kind` (`Kind`), `amount` (`float`); method
  `signed_amount(self) -> float` returns `+amount` when
  `self.kind is Kind.INCOME`, otherwise `-amount`.

Two ideas in one file: the Enum gives a **fixed-set** `kind`, and
`frozen=True` makes the record immutable → **hashable** → usable as a
dictionary key (the DSA #1 hash-map idea, now at the type level).

**Constraints:**
- Compare with `is` against `Kind` members, not string literals.
- No assignment after construction — a frozen record is a value.

**File tree:**
```text
kind.py             <- you create
```

**Driver:**
```python
from kind import Kind, Transaction

print(Kind.INCOME.value)
print(Kind.EXPENSE.value)
print([k.name for k in Kind])

pay = Transaction("2026-09-25", "salary", Kind.INCOME, 100.0)
buy = Transaction("2026-09-04", "food", Kind.EXPENSE, 12.5)
print(pay.signed_amount())
print(buy.signed_amount())
print(pay.kind is Kind.INCOME)
print(pay == Transaction("2026-09-25", "salary", Kind.INCOME, 100.0))
print(pay is Transaction("2026-09-25", "salary", Kind.INCOME, 100.0))

lookup = {pay: "salary day"}
print(lookup[Transaction("2026-09-25", "salary", Kind.INCOME, 100.0)])
```

**Output:**
```
income
expense
['INCOME', 'EXPENSE']
100.0
-12.5
True
True
False
salary day
```

In [3]:
from kind import Kind, Transaction

print(Kind.INCOME.value)
print(Kind.EXPENSE.value)
print([k.name for k in Kind])

pay = Transaction("2026-09-25", "salary", Kind.INCOME, 100.0)
buy = Transaction("2026-09-04", "food", Kind.EXPENSE, 12.5)
print(pay.signed_amount())
print(buy.signed_amount())
print(pay.kind is Kind.INCOME)
print(pay == Transaction("2026-09-25", "salary", Kind.INCOME, 100.0))
print(pay is Transaction("2026-09-25", "salary", Kind.INCOME, 100.0))

lookup = {pay: "salary day"}
print(lookup[Transaction("2026-09-25", "salary", Kind.INCOME, 100.0)])

income
expense
['INCOME', 'EXPENSE']
100.0
-12.5
True
True
False
salary day


## M1 — composition: a `Ledger` dataclass holding records

**Type:** [REAL-WORLD] · medium

**Task:** create `ledger.py` that composes your E3 file:

- `@dataclass class Ledger` — fields `owner` (`str`) and
  `transactions` (`list[Transaction]`, `field(default_factory=list)`).
- Method `add(self, txn: Transaction) -> None` — appends to the list.
- Method `net(self) -> float` — `sum(txn.signed_amount() for txn in self.transactions)`.
- Method `report(self) -> str` — first line `f"{owner}: {net:.2f}"`, then
  one line per transaction **sorted by date** ascending, formatted as
  `f"{date} {category} {signed_amount:+.2f}"`.

Trace `Ledger.net()` through the layers (the Lesson 15 reflection
skill): the ledger asks *each* `Transaction` to sign itself. The data
classes make the object graph visible in `print(book)` — a single
multi-layer object instead of scattered dicts.

**Constraints:**
- `from kind import Transaction, Kind` — reuse E3's frozen record.
- All methods annotated; `add` returns `None`; `report` returns the
  string, it does not print.

**File tree:**
```text
ledger.py           <- you create
kind.py             <- from E3 (already created)
```

**Driver:**
```python
from ledger import Ledger
from kind import Kind, Transaction

book = Ledger("ada")
book.add(Transaction("2026-09-04", "food", Kind.EXPENSE, 12.5))
book.add(Transaction("2026-09-25", "salary", Kind.INCOME, 100.0))
book.add(Transaction("2026-10-01", "food", Kind.EXPENSE, 8.0))

print(book)
print(book.net())
print(book.report())
```

**Output:**
```
Ledger(owner='ada', transactions=[Transaction(date='2026-09-04', category='food', kind=<Kind.EXPENSE: 'expense'>, amount=12.5), Transaction(date='2026-09-25', category='salary', kind=<Kind.INCOME: 'income'>, amount=100.0), Transaction(date='2026-10-01', category='food', kind=<Kind.EXPENSE: 'expense'>, amount=8.0)])
79.5
ada: 79.50
2026-09-04 food -12.50
2026-09-25 salary +100.00
2026-10-01 food -8.00
```

In [1]:
from ledger import Ledger
from kind import Kind, Transaction

book = Ledger("ada")
book.add(Transaction("2026-09-04", "food", Kind.EXPENSE, 12.5))
book.add(Transaction("2026-09-25", "salary", Kind.INCOME, 100.0))
book.add(Transaction("2026-10-01", "food", Kind.EXPENSE, 8.0))

print(book)
print(book.net())
print(book.report())

Ledger(owner='ada', transactions=[Transaction(date='2026-09-04', category='food', kind=<Kind.EXPENSE: 'expense'>, amount=12.5), Transaction(date='2026-09-25', category='salary', kind=<Kind.INCOME: 'income'>, amount=100.0), Transaction(date='2026-10-01', category='food', kind=<Kind.EXPENSE: 'expense'>, amount=8.0)])
79.5
ada: 79.50
2026-09-04 food -12.50
2026-09-25 salary +100.00
2026-10-01 food -8.00


## M2 — `asdict()` + JSON round trip with an Enum-aware encoder

**Type:** [REAL-WORLD] · medium

**Task:** create `serialize.py` with two functions:

- `kind_encoder(obj)` — returns `obj.value` when `obj` is a `Kind`
  member, otherwise raises `TypeError` (this is the `default=` hook
  `json.dumps` calls for values it cannot serialize).
- `save_json(path: Path, transactions: list[Transaction]) -> None` —
  writes `[asdict(t) for t in transactions]` with `json.dumps` using
  `indent=2` and `default=kind_encoder` (Lesson 11 discipline:
  `Path.write_text(..., encoding="utf-8")`).

The wrinkle: `asdict` recurses into dataclasses, but a `Kind` member
inside a dict is still `<Kind.EXPENSE: 'expense'>` — not JSON-able. The
encoder maps it to `"expense"`, so storage stays byte-identical to
Lesson 15 while code compares with `is`. Reading back is the reverse:
`Kind(raw["kind"])` re-wraps the stored string.

**Constraints:**
- `from dataclasses import asdict`; `from kind import Kind, Transaction`.
- Functions are annotated; `save_json` returns `None`.
- The driver creates `serialized_out.json` in the lesson folder — that
  is expected; delete it before committing.

**File tree:**
```text
serialize.py        <- you create
kind.py             <- from E3 (already created)
```

**Driver:**
```python
import json
from pathlib import Path
from kind import Kind, Transaction
from serialize import save_json

txs = [
    Transaction("2026-09-04", "food", Kind.EXPENSE, 12.5),
    Transaction("2026-09-25", "salary", Kind.INCOME, 100.0),
]

save_json(Path("serialized_out.json"), txs)

data = json.loads(Path("serialized_out.json").read_text(encoding="utf-8"))
print(type(data).__name__)
print(data[1]["kind"])
print(data[0] == {"date": "2026-09-04", "category": "food", "kind": "expense", "amount": 12.5})
print(data)

raw = data[0]
back = Transaction(raw["date"], raw["category"], Kind(raw["kind"]), raw["amount"])
print(back == txs[0])
print(back)
```

**Output:**
```
list
income
True
[{'date': '2026-09-04', 'category': 'food', 'kind': 'expense', 'amount': 12.5}, {'date': '2026-09-25', 'category': 'salary', 'kind': 'income', 'amount': 100.0}]
True
Transaction(date='2026-09-04', category='food', kind=<Kind.EXPENSE: 'expense'>, amount=12.5)
```

In [1]:
import json
from pathlib import Path
from kind import Kind, Transaction
from serialize import save_json

txs = [
    Transaction("2026-09-04", "food", Kind.EXPENSE, 12.5),
    Transaction("2026-09-25", "salary", Kind.INCOME, 100.0),
]

save_json(Path("serialized_out.json"), txs)

data = json.loads(Path("serialized_out.json").read_text(encoding="utf-8"))
print(type(data).__name__)
print(data[1]["kind"])
print(data[0] == {"date": "2026-09-04", "category": "food", "kind": "expense", "amount": 12.5})
print(data)

raw = data[0]
back = Transaction(raw["date"], raw["category"], Kind(raw["kind"]), raw["amount"])
print(back == txs[0])
print(back)

list
income
True
[{'date': '2026-09-04', 'category': 'food', 'kind': 'expense', 'amount': 12.5}, {'date': '2026-09-25', 'category': 'salary', 'kind': 'income', 'amount': 100.0}]
True
Transaction(date='2026-09-04', category='food', kind=<Kind.EXPENSE: 'expense'>, amount=12.5)


## H1 — debug drill: a dataclass budget that lies

**Type:** [REAL-WORLD] · hard

**Scenario:** a `budget` module should load a JSON log and print the
owner and net total when you run `python main_h1.py`. It fails at
import, or crashes, or prints the wrong numbers. **Four** bugs are
planted — one import-time, one annotation, one runtime, one logic. Find
and fix all four with the DEBUGGING protocol (`notes/DEBUGGING.md`).
After each fix, update your bug report in `solution.py`.

**Fixture** (`budget.json` — create exactly this):
```json
[
  {"date": "2026-09-04", "category": "food",   "kind": "expense", "amount": 12.5},
  {"date": "2026-09-25", "category": "salary", "kind": "income",  "amount": 100.0},
  {"date": "2026-10-01", "category": "food",   "kind": "expense", "amount": 8.0}
]
```

**File tree (create these files WITH the bugs — do not fix while
typing):**
```text
budget.py           <- you create
budget.json         <- the fixture above
main_h1.py          <- you create
```

**`budget.py`: (copy exactly — the bugs are planted in it)**
```python
from dataclasses import dataclass, field
from enum import Enum
import json
from pathlib import Path


class Kind(Enum):
    INCOME = "income"
    EXPENSE = "expense"


@dataclass(frozen=True)
class Transaction:
    date: str
    category: str
    kind: Kind
    amount: str

    def signed_amount(self) -> float:
        if self.kind == "income":
            return self.amount
        return -self.amount

    def mark_spent(self, extra: float) -> None:
        self.amount = extra


@dataclass
class Budget:
    owner: str
    transactions: list[Transaction] = []

    def add(self, txn: Transaction) -> None:
        self.transactions.append(txn)

    def net(self) -> float:
        return sum(txn.signed_amount() for txn in self.transactions)
```

**`main_h1.py`:**
```python
from pathlib import Path
import json
from budget import Budget, Kind, Transaction


def main() -> None:
    records = json.loads(Path("budget.json").read_text(encoding="utf-8"))
    book = Budget("ada")
    for record in records:
        txn = Transaction(
            record["date"], record["category"], Kind(record["kind"]), record["amount"]
        )
        txn.mark_spent(txn.amount)
        book.add(txn)
    print(f"owner: {book.owner}")
    print(f"net: {book.net():.2f}")


if __name__ == "__main__":
    main()
```

**Constraints:**
- Fix the code; do not change the expected result, the fixture, or the
  printed lines.
- After all fixes, every field and method carries honest annotations;
  `signed_amount` compares with `is`; `mark_spent` returns a **new**
  `Transaction` (a frozen record cannot be mutated — replace, don't
  edit) and `main_h1.py` updates the variable in place.
- Stretch (this is the Lesson 15 lesson): run `python -m mypy main_h1.py`
  — it must report no errors, which catches the annotation lie if the
  runtime checks missed it.

**Expected result after all fixes:** running `python main_h1.py` prints
exactly:
```
owner: ada
net: 79.50
```

**Bug report:** append to `solution.py` — for each of the four bugs:
expected vs observed and the root cause.

**Driver** (run from the lesson folder):
```python
import subprocess, sys

result = subprocess.run(
    [sys.executable, "main_h1.py"],
    capture_output=True,
    text=True,
)
print("returncode:", result.returncode)
print("stdout:", repr(result.stdout))
print("stderr:", repr(result.stderr))
```

**Output (after the fixes):**
```
returncode: 0
stdout: 'owner: ada\nnet: 79.50\n'
stderr: ''
```

In [2]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "main_h1.py"],
    capture_output=True,
    text=True,
)
print("returncode:", result.returncode)
print("stdout:", repr(result.stdout))
print("stderr:", repr(result.stderr))

returncode: 0
stdout: 'owner: ada\nnet: 79.50\n'
stderr: ''


## Challenge — refactor the Lesson 15 Expense Manager onto the frozen model

**Type:** [REAL-WORLD] · micro build

**Task:** rebuild the Lesson 15 Challenge package so the data model is
the frozen dataclass from E3. The **interfaces stay identical** to
Lesson 15 — `load.py`, `compute.py`, `write.py`, `main.py` keep the
same function names, signatures, and printed output. Only the model
underneath changes: a hand-written class carrying strings becomes a
`frozen` dataclass with a `Kind` enum field. This is the WP2-ready
model.

**Fixture** (`transactions.json` — create exactly this):
```json
[
  {"date": "2026-09-04", "category": "food",   "kind": "expense", "amount": 12.5},
  {"date": "2026-09-18", "category": "ride",   "kind": "expense", "amount": 4.5},
  {"date": "2026-09-25", "category": "salary", "kind": "income",  "amount": 100.0},
  {"date": "2026-10-01", "category": "food",   "kind": "expense", "amount": 8.0}
]
```

**File tree (all files you create):**
```text
expense/
    __init__.py       <- empty
    load.py           <- you create
    compute.py        <- you create
    write.py          <- you create
transactions.json     <- the fixture above
main.py               <- at the lesson root, OUTSIDE the package
```

**Interface contract (identical to Lesson 15, return annotations as
stated):**

- `expense/load.py` — `load_transactions(path: Path) -> list[Transaction]`
  reads the fixture and builds one frozen `Transaction` per record,
  importing it with `from kind import Transaction, Kind` and wrapping
  `Kind(record["kind"])`. Missing/broken file → `[]` (L11 loader
  contract).
- `expense/compute.py` —
  - `net_by_month(transactions: list[Transaction]) -> dict[str, float]`:
    month = `txn.date[:7]`; add each `txn.signed_amount()` with the
    `.get(month, 0.0)` accumulate pattern.
  - `net_by_category(transactions: list[Transaction]) -> dict[str, float]`:
    same logic grouped by `txn.category`.
- `expense/write.py` — `write_summary(path: Path, totals: dict[str, float], colname: str) -> None`
  writes a CSV with header `{colname},total`, one row per key sorted
  alphabetically, totals with two decimals, opened with
  `Path.open(..., "w", newline="", encoding="utf-8")` (L12).
- `main.py` — `main() -> None`: loads, computes both summaries, writes
  `monthly.csv` and `categories.csv` (L12), then prints the month lines
  and the category lines (`key: total`, two decimals), months and
  categories both **sorted alphabetically**:
```text
2026-09: 83.00
2026-10: -8.00
food: -20.50
ride: -4.50
salary: 100.00
```

**Constraints:**
- Every function is annotated as stated. `signed_amount` does the
  signing — `compute` never reimplements it.
- Loaders follow the L9/L11 contract; CSV writing follows L12 rules.
- No step-by-step plan: decide the composition yourself. This is a
  **refactor** — copy the shape you already built in Lesson 15, swap the
  model, rerun everything.

**Driver** (run from the lesson folder):
```python
import subprocess, sys

result = subprocess.run(
    [sys.executable, "main.py"],
    capture_output=True,
    text=True,
)
print("returncode:", result.returncode)
print("stdout:", repr(result.stdout))
print("stderr:", repr(result.stderr))
```

**Output:**
```
returncode: 0
stdout: '2026-09: 83.00\n2026-10: -8.00\nfood: -20.50\nride: -4.50\nsalary: 100.00\n'
stderr: ''
```

**Stretch 1 (mandatory-adjacent — the L14/L15 habit):** run
```
python -m mypy main.py
```
`mypy` should report no errors.

**Stretch 2 (AWARENESS — `slots`):** switch the frozen dataclass to
`@dataclass(frozen=True, slots=True)` and rerun the driver. The output
must not change — if it does, the class depends on ad-hoc attributes.
This is the compact-storage win for bulk records. Ask before installing
anything.

In [3]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "main.py"],
    capture_output=True,
    text=True,
)
print("returncode:", result.returncode)
print("stdout:", repr(result.stdout))
print("stderr:", repr(result.stderr))

returncode: 0
stdout: '2026-09: 83.00\n2026-10: -8.00\nfood: -20.50\nride: -4.50\nsalary: 100.00\n'
stderr: ''
